# 01 — Bronze: EIA and Census ingestion

Lands real API responses for the latest **eight complete monthly periods common to both EIA
datasets**. The notebook fails honestly when its EIA key or Key Vault settings are absent.
It never creates sample rows.

**Sources**

* EIA v2 `electricity/electric-power-operational-data` — monthly state/fuel generation,
  consumption-for-electricity-generation, and Btu fields.
* EIA v2 `electricity/retail-sales` — monthly state retail sales MWh.
* Census Population Estimates API — Vintage 2025 state population estimate.

Bind this notebook to a schema-enabled Lakehouse before running. In production, pass the
parameters from a Fabric Variable Library or pipeline; do not place keys in notebook source.

In [ ]:
environment = "dev"  # dev | test | prod
key_vault_url = ""  # e.g. https://<vault>.vault.azure.net/
eia_secret_name = "eia-api-key"
census_secret_name = "census-api-key"
eia_base_url = "https://api.eia.gov/v2"
census_population_url = "https://api.census.gov/data/2025/pep/population"
census_population_variable = "POP_2025"
expected_jurisdictions = 51
page_size = 5000

In [ ]:
from datetime import datetime, timezone
import json
import time
import uuid
from urllib.parse import urlencode

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import notebookutils
from pyspark.sql import functions as F, types as T

if environment not in {"dev", "test", "prod"}:
    raise ValueError("environment must be dev, test, or prod")
if not key_vault_url:
    raise RuntimeError(
        "key_vault_url is required. Configure it through a Fabric Variable Library or pipeline parameter."
    )
try:
    eia_api_key = notebookutils.credentials.getSecret(key_vault_url, eia_secret_name)
except Exception as exc:
    raise RuntimeError(
        f"Unable to retrieve EIA key '{eia_secret_name}' from Key Vault. "
        "Grant the notebook identity secret access and verify runtime configuration."
    ) from exc
if not eia_api_key:
    raise RuntimeError("The configured EIA secret is empty.")
try:
    census_api_key = notebookutils.credentials.getSecret(key_vault_url, census_secret_name)
except Exception as exc:
    raise RuntimeError(
        f"Unable to retrieve Census key '{census_secret_name}' from Key Vault. "
        "Grant the notebook identity secret access and verify runtime configuration."
    ) from exc
if not census_api_key:
    raise RuntimeError("The configured Census API secret is empty.")

batch_id = str(uuid.uuid4())
retrieved_at = datetime.now(timezone.utc)
session = requests.Session()
session.mount(
    "https://",
    HTTPAdapter(max_retries=Retry(total=5, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504])),
)
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

## Robust EIA paging and metadata-driven period selection

Every page is checked against EIA's reported `total`. The period probe requests real rows from
each route, then intersects valid `YYYY-MM` periods. A period cannot enter the processing window
unless it exists in **both** EIA sources.

In [ ]:
def eia_get(route, params):
    url = f"{eia_base_url}/{route.strip('/')}/data/"
    response = session.get(url, params=params, timeout=90)
    response.raise_for_status()
    payload = response.json()
    if "error" in payload:
        raise RuntimeError(f"EIA returned an error for {route}: {payload['error']}")
    return url, payload


def eia_pages(route, data_fields, extra_params=None):
    offset = 0
    rows = []
    source_url = None
    response_metadata = None
    while True:
        params = [
            ("api_key", eia_api_key),
            ("frequency", "monthly"),
            ("offset", str(offset)),
            ("length", str(page_size)),
            ("sort[0][column]", "period"),
            ("sort[0][direction]", "asc"),
        ]
        params.extend((f"data[{index}]", field) for index, field in enumerate(data_fields))
        params.extend(extra_params or [])
        source_url, payload = eia_get(route, params)
        response = payload.get("response") or {}
        page = response.get("data") or []
        rows.extend(page)
        response_metadata = {key: value for key, value in response.items() if key != "data"}
        total = int(response.get("total") or 0)
        offset += len(page)
        if not page or offset >= total:
            break
        time.sleep(0.15)
    if response_metadata is None:
        raise RuntimeError(f"EIA route {route} returned no response metadata.")
    if len(rows) != int(response_metadata.get("total") or len(rows)):
        raise RuntimeError(f"Incomplete EIA pagination for {route}: {len(rows)} rows retrieved.")
    return source_url, response_metadata, rows


def metadata_end_period(route):
    url = f"{eia_base_url}/{route.strip('/')}/"
    response = session.get(url, params={"api_key": eia_api_key}, timeout=90)
    response.raise_for_status()
    payload = response.json()
    monthly_periods = []

    def visit(node):
        if isinstance(node, dict):
            for key, value in node.items():
                if (
                    key == "endPeriod"
                    and isinstance(value, str)
                    and len(value) == 7
                    and value[4] == "-"
                ):
                    monthly_periods.append(value)
                visit(value)
        elif isinstance(node, list):
            for value in node:
                visit(value)

    visit(payload)
    if not monthly_periods:
        raise RuntimeError(f"EIA metadata for {route} has no monthly endPeriod.")
    return max(monthly_periods)


def month_sequence(end_period, count):
    year, month = (int(part) for part in end_period.split("-"))
    values = []
    for _ in range(count):
        values.append(f"{year:04d}-{month:02d}")
        month -= 1
        if month == 0:
            month = 12
            year -= 1
    return sorted(values)


operational_end = metadata_end_period("electricity/electric-power-operational-data")
retail_end = metadata_end_period("electricity/retail-sales")
common_end = min(operational_end, retail_end)
selected_periods = month_sequence(common_end, 8)
window_params = [("start", selected_periods[0]), ("end", selected_periods[-1])]
print(
    {
        "operational_end_period": operational_end,
        "retail_end_period": retail_end,
        "selected_common_periods": selected_periods,
    }
)

## Fetch and stage source rows

The operational query uses EIA's all-electric-power-sector facet (`99`) and keeps state and fuel
dimensions. Retail sales uses the all-sector record. Raw strings are retained so suppression
markers remain distinguishable from missing values in Silver.

In [ ]:
operational_fields = [
    "generation",
    "total-consumption",
    "consumption-for-eg",
    "total-consumption-btu",
    "consumption-for-eg-btu",
]
op_url, op_metadata, operational_rows = eia_pages(
    "electricity/electric-power-operational-data",
    operational_fields,
    window_params + [("facets[sectorid][]", "99")],
)
retail_url, retail_metadata, retail_rows = eia_pages(
    "electricity/retail-sales",
    ["sales"],
    window_params + [("facets[sectorid][]", "ALL")],
)
for name, rows in {
    "operational": operational_rows,
    "retail_sales": retail_rows,
}.items():
    returned_periods = {str(row.get("period")) for row in rows}
    missing_periods = sorted(set(selected_periods) - returned_periods)
    if missing_periods:
        raise RuntimeError(
            f"{name} metadata advertised the selected window but returned no rows for {missing_periods}."
        )

census_params = {
    "get": f"NAME,{census_population_variable}",
    "for": "state:*",
    "key": census_api_key,
}
census_response = session.get(census_population_url, params=census_params, timeout=90)
census_response.raise_for_status()
census_payload = census_response.json()
if not isinstance(census_payload, list) or len(census_payload) < 2:
    raise RuntimeError("Census Vintage 2025 endpoint returned no state population rows.")
headers = census_payload[0]
if census_population_variable not in headers:
    raise RuntimeError(
        f"Census response does not contain {census_population_variable}; "
        "do not substitute an older vintage silently."
    )
census_rows = [dict(zip(headers, row)) for row in census_payload[1:]]

In [ ]:
raw_schema = T.StructType(
    [
        T.StructField("dataset_name", T.StringType(), False),
        T.StructField("period", T.StringType(), True),
        T.StructField("state_code_raw", T.StringType(), True),
        T.StructField("state_name_raw", T.StringType(), True),
        T.StructField("state_fips_raw", T.StringType(), True),
        T.StructField("sector_code_raw", T.StringType(), True),
        T.StructField("fuel_code_raw", T.StringType(), True),
        T.StructField("fuel_name_raw", T.StringType(), True),
        T.StructField("generation_raw", T.StringType(), True),
        T.StructField("total_consumption_raw", T.StringType(), True),
        T.StructField("consumption_for_eg_raw", T.StringType(), True),
        T.StructField("total_consumption_btu_raw", T.StringType(), True),
        T.StructField("consumption_for_eg_btu_raw", T.StringType(), True),
        T.StructField("retail_sales_raw", T.StringType(), True),
        T.StructField("population_raw", T.StringType(), True),
        T.StructField("generation_units_raw", T.StringType(), True),
        T.StructField("consumption_for_eg_btu_units_raw", T.StringType(), True),
        T.StructField("retail_sales_units_raw", T.StringType(), True),
        T.StructField("source_url", T.StringType(), False),
        T.StructField("source_units_json", T.StringType(), True),
        T.StructField("source_metadata_json", T.StringType(), True),
        T.StructField("batch_id", T.StringType(), False),
        T.StructField("ingested_at_utc", T.TimestampType(), False),
        T.StructField("ingestion_date", T.DateType(), False),
    ]
)

def as_text(value):
    return None if value is None else str(value)

def op_record(row):
    return (
        "eia_electric_power_operational",
        as_text(row.get("period")),
        as_text(row.get("location") or row.get("stateid")),
        as_text(row.get("stateDescription")),
        None,
        as_text(row.get("sectorid")),
        as_text(row.get("fueltypeid")),
        as_text(row.get("fuelTypeDescription")),
        as_text(row.get("generation")),
        as_text(row.get("total-consumption")),
        as_text(row.get("consumption-for-eg")),
        as_text(row.get("total-consumption-btu")),
        as_text(row.get("consumption-for-eg-btu")),
        None,
        None,
        as_text(row.get("generation-units")),
        as_text(row.get("consumption-for-eg-btu-units")),
        None,
        op_url,
        json.dumps(op_metadata.get("units", {}), sort_keys=True),
        json.dumps(op_metadata, sort_keys=True),
        batch_id,
        retrieved_at,
        retrieved_at.date(),
    )

def retail_record(row):
    return (
        "eia_retail_sales", as_text(row.get("period")),
        as_text(row.get("stateid") or row.get("location")),
        as_text(row.get("stateDescription")), None, as_text(row.get("sectorid")),
        None, None, None, None, None, None, None, as_text(row.get("sales")), None,
        None, None, as_text(row.get("sales-units")),
        retail_url, json.dumps(retail_metadata.get("units", {}), sort_keys=True),
        json.dumps(retail_metadata, sort_keys=True), batch_id, retrieved_at, retrieved_at.date(),
    )

def census_record(row):
    return (
        "census_population_vintage_2025", None, None, as_text(row.get("NAME")),
        as_text(row.get("state")), None, None, None, None, None, None, None, None,
        None, as_text(row.get(census_population_variable)),
        None, None, None,
        census_response.url, json.dumps({census_population_variable: "persons"}),
        json.dumps({"vintage": "2025", "variable": census_population_variable}),
        batch_id, retrieved_at, retrieved_at.date(),
    )

raw_records = (
    [op_record(row) for row in operational_rows]
    + [retail_record(row) for row in retail_rows]
    + [census_record(row) for row in census_rows]
)
bronze_df = spark.createDataFrame(raw_records, raw_schema)

landing_path = f"Files/energy/bronze/{environment}/{retrieved_at:%Y/%m/%d}/{batch_id}"
notebookutils.fs.mkdirs(landing_path)
notebookutils.fs.put(
    f"{landing_path}/manifest.json",
    json.dumps(
        {
            "batch_id": batch_id,
            "retrieved_at_utc": retrieved_at.isoformat(),
            "selected_periods": selected_periods,
            "row_counts": {
                "operational": len(operational_rows),
                "retail_sales": len(retail_rows),
                "census_population": len(census_rows),
            },
        },
        indent=2,
    ),
    True,
)

(bronze_df.write.format("delta").mode("append").partitionBy("ingestion_date", "dataset_name")
 .saveAsTable("bronze.energy_source_raw"))

In [ ]:
counts = {row["dataset_name"]: row["count"] for row in bronze_df.groupBy("dataset_name").count().collect()}
required = {
    "eia_electric_power_operational",
    "eia_retail_sales",
    "census_population_vintage_2025",
}
if set(counts) != required or any(counts[name] == 0 for name in required):
    raise RuntimeError(f"Bronze validation failed; source counts: {counts}")
if set(bronze_df.where(F.col("period").isNotNull()).select("period").distinct().toPandas()["period"]) != set(selected_periods):
    raise RuntimeError("Bronze periods do not exactly match the selected common window.")
print({"batch_id": batch_id, "counts": counts, "selected_periods": selected_periods})